In [ ]:
# ---- CELL 1: Mount Drive ----
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# ---- CELL 2: Install ffmpeg (usually already present, this just confirms) ----
!apt-get -qq install -y ffmpeg


In [ ]:
# ---- CELL 3: Settings — EDIT THESE PATHS TO MATCH YOUR DRIVE ----

# Folder on your Drive containing the 29 raw .MOV videos
RAW_VIDEO_DIR = "/content/drive/MyDrive/CourtVision/raw_shots_footage"

# Path to your filled shot-log spreadsheet (upload it into this Drive folder,
# or point this at wherever you keep it)
SHOT_LOG_PATH = "/content/drive/MyDrive/CourtVision/courtvision_shot_log.xlsx"

# Where cut clips will be written (created automatically)
OUTPUT_DIR = "/content/drive/MyDrive/CourtVision/clips"

# Padding around the logged release timestamp, in seconds
PRE_PAD = 2.0   # seconds before release (covers wind-up motion)
POST_PAD = 2.0  # seconds after release (follow-through)


In [ ]:
# Copy raw videos from Drive to local Colab disk (much faster for ffmpeg to read)
import shutil, os

LOCAL_RAW_DIR = "/content/raw_footage_local"
os.makedirs(LOCAL_RAW_DIR, exist_ok=True)

for fname in os.listdir(RAW_VIDEO_DIR):
    src = os.path.join(RAW_VIDEO_DIR, fname)
    dst = os.path.join(LOCAL_RAW_DIR, fname)
    if not os.path.exists(dst):
        shutil.copy2(src, dst)
        print("copied", fname)

RAW_VIDEO_DIR = LOCAL_RAW_DIR  # point the rest of the script at the local copy

In [ ]:
# ---- CELL 4: Install/import openpyxl + ffmpeg-python helpers ----
!pip install -q openpyxl

import os
import subprocess
import openpyxl

os.makedirs(os.path.join(OUTPUT_DIR, "2pt"), exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, "3pt"), exist_ok=True)


def normalize_label(raw_label):
    """Turns '2pt', '2 pt', '2PT', ' 3pt ' etc. into a clean '2pt' / '3pt'."""
    if raw_label is None:
        return None
    cleaned = str(raw_label).strip().lower().replace(" ", "")
    if cleaned in ("2pt", "2points", "2point"):
        return "2pt"
    if cleaned in ("3pt", "3points", "3point"):
        return "3pt"
    return None  # unrecognized label — will be flagged, not silently guessed


def get_video_duration(video_path):
    """Uses ffprobe to get video duration in seconds."""
    result = subprocess.run(
        [
            "ffprobe", "-v", "error", "-show_entries", "format=duration",
            "-of", "default=noprint_wrappers=1:nokey=1", video_path,
        ],
        stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True,
    )
    try:
        return float(result.stdout.strip())
    except ValueError:
        return None


def cut_clip(video_path, start_sec, end_sec, output_path):
    """Cuts [start_sec, end_sec) from video_path into output_path, re-encoded
    (not stream-copied) so the cut lands on an exact frame boundary."""
    duration = end_sec - start_sec
    cmd = [
        "ffmpeg", "-y",
        "-ss", str(start_sec),
        "-i", video_path,
        "-t", str(duration),
        "-c:v", "libx264", "-preset", "fast", "-crf", "20",
        "-c:a", "aac",
        output_path,
    ]
    result = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    return result.returncode == 0, result.stderr


In [ ]:

# ---- CELL 5: Read the shot log ----

wb = openpyxl.load_workbook(SHOT_LOG_PATH, data_only=True)
ws = wb.active

# Data starts at row 3 (row 1 = title banner, row 2 = headers, row 3 = example row)
rows = list(ws.iter_rows(min_row=3, max_row=ws.max_row, values_only=True))

shots = []  # list of dicts: video, shot_number, timestamp, label
skipped_blank = 0
skipped_bad_label = 0

for video, shot_number, timestamp_sec, label in rows:
    if video is None:
        continue
    if timestamp_sec in (None, "-", "") or not isinstance(timestamp_sec, (int, float)):
        skipped_blank += 1
        continue
    clean_label = normalize_label(label)
    if clean_label is None:
        print(f"⚠️  Unrecognized label '{label}' for {video} shot {shot_number} — skipped")
        skipped_bad_label += 1
        continue
    shots.append({
        "video": video,
        "shot_number": shot_number,
        "timestamp": float(timestamp_sec),
        "label": clean_label,
    })

print(f"Loaded {len(shots)} shots to cut")
print(f"Skipped {skipped_blank} blank rows, {skipped_bad_label} unrecognized labels")

Loaded 116 shots to cut
Skipped 30 blank rows, 0 unrecognized labels


In [ ]:
import time

import shlex

def cut_clip_with_timeout(video_path, start_sec, end_sec, output_path, timeout=90):
    duration = end_sec - start_sec
    cmd = (
        f'timeout {timeout} ffmpeg -y -nostdin -hide_banner -loglevel error '
        f'-ss {start_sec} -i {shlex.quote(video_path)} -t {duration} '
        f'-vf "scale=640:-2" '
        f'-c:v libx264 -preset ultrafast -crf 23 -c:a aac {shlex.quote(output_path)}'
    )
    raw = os.system(cmd)
    exit_code = os.waitstatus_to_exitcode(raw) if hasattr(os, "waitstatus_to_exitcode") else (raw >> 8)

    if exit_code == 0:
        return True, ""
    elif exit_code == 124:
        return False, f"TIMED OUT after {timeout}s"
    else:
        return False, f"ffmpeg exited with code {exit_code}"


success_count = 0
fail_log = []

for i, shot in enumerate(shots, start=1):
    video_filename = shot["video"]
    video_path = os.path.join(RAW_VIDEO_DIR, video_filename)

    print(f"[{i}/{len(shots)}] {video_filename} shot {shot['shot_number']} ({shot['label']})...", flush=True)

    if not os.path.exists(video_path):
        print("   -> MISSING FILE, skipped")
        fail_log.append(f"Missing video file: {video_path}")
        continue

    duration = get_video_duration(video_path)
    if duration is None:
        print("   -> could not read duration, skipped")
        fail_log.append(f"Could not read duration for {video_filename}")
        continue

    release_ts = shot["timestamp"]
    start = max(0.0, release_ts - PRE_PAD)
    end = min(duration, release_ts + POST_PAD)

    if end <= start:
        print("   -> bad time window, skipped")
        fail_log.append(f"Bad window for {video_filename} shot {shot['shot_number']}")
        continue

    video_stem = os.path.splitext(video_filename)[0]
    out_name = f"{video_stem}_shot{shot['shot_number']}_{shot['label']}.mp4"
    out_path = os.path.join(OUTPUT_DIR, shot["label"], out_name)

    t0 = time.time()
    ok, err = cut_clip_with_timeout(video_path, start, end, out_path, timeout=90)
    elapsed = time.time() - t0

    if ok:
        print(f"   -> done in {elapsed:.1f}s")
        success_count += 1
    else:
        print(f"   -> FAILED after {elapsed:.1f}s: {err[-200:]}")
        fail_log.append(f"{video_filename} shot {shot['shot_number']}: {err[-300:]}")

print(f"\n✅ Cut {success_count} / {len(shots)} clips successfully")
if fail_log:
    print(f"\n⚠️ {len(fail_log)} failures:")
    for line in fail_log:
        print(" -", line)

[1/116] IMG_3128.MOV shot 1 (2pt)...
   -> done in 13.4s
[2/116] IMG_3128.MOV shot 2 (2pt)...
   -> done in 18.5s
[3/116] IMG_3128.MOV shot 3 (2pt)...
   -> done in 12.4s
[4/116] IMG_3127.MOV shot 1 (2pt)...
   -> done in 11.9s
[5/116] IMG_3127.MOV shot 2 (2pt)...
   -> done in 13.8s
[6/116] IMG_3127.MOV shot 3 (2pt)...
   -> done in 14.5s
[7/116] IMG_3127.MOV shot 4 (2pt)...
   -> done in 14.3s
[8/116] IMG_3126.MOV shot 1 (2pt)...
   -> done in 12.1s
[9/116] IMG_3126.MOV shot 2 (2pt)...
   -> done in 11.4s
[10/116] IMG_3126.MOV shot 3 (2pt)...
   -> done in 13.6s
[11/116] IMG_3126.MOV shot 4 (2pt)...
   -> done in 13.8s
[12/116] IMG_3126.MOV shot 5 (2pt)...
   -> done in 14.1s
[13/116] IMG_3124.MOV shot 1 (2pt)...
   -> done in 13.6s
[14/116] IMG_3124.MOV shot 2 (2pt)...
   -> done in 12.2s
[15/116] IMG_3124.MOV shot 3 (2pt)...
   -> done in 12.3s
[16/116] IMG_3124.MOV shot 4 (2pt)...
   -> done in 14.1s
[17/116] IMG_3124.MOV shot 5 (2pt)...
   -> done in 14.2s
[18/116] IMG_3121.MOV s

In [ ]:
# # # print("RAW_VIDEO_DIR is:", RAW_VIDEO_DIR)
# # # test_path = os.path.join(RAW_VIDEO_DIR, "IMG_3128.MOV")
# # # print("Exists:", os.path.exists(test_path))
# # # print("Size (MB):", os.path.getsize(test_path) / 1e6 if os.path.exists(test_path) else "N/A")

# # test_video = os.path.join(RAW_VIDEO_DIR, "IMG_3128.MOV")
# # os.system(f'ffmpeg -y -nostdin -ss 2 -i "{test_video}" -t 4 -c:v libx264 -preset fast -crf 20 -c:a aac /content/test_clip.mp4')

# import os
# for label in ("2pt", "3pt"):
#     path = f"/content/clips_local/{label}"
#     if os.path.exists(path):
#         print(label, "->", len(os.listdir(path)), "clips found")
#     else:
#         print(label, "-> folder missing")

In [ ]:
import os, glob

print("OUTPUT_DIR is:", OUTPUT_DIR)
print("LOCAL_OUTPUT_DIR defined:", "LOCAL_OUTPUT_DIR" in globals())

for name, base in [
    ("OUTPUT_DIR (what Cell 6 wrote to)", OUTPUT_DIR),
    ("local /content/clips_local", "/content/clips_local"),
    ("Drive clips folder", "/content/drive/MyDrive/CourtVision/clips"),
]:
    for label in ("2pt", "3pt"):
        files = glob.glob(os.path.join(base, label, "*.mp4"))
        print(f"{name} | {label}: {len(files)} clips")

OUTPUT_DIR is: /content/drive/MyDrive/CourtVision/clips
LOCAL_OUTPUT_DIR defined: False
OUTPUT_DIR (what Cell 6 wrote to) | 2pt: 63 clips
OUTPUT_DIR (what Cell 6 wrote to) | 3pt: 53 clips
local /content/clips_local | 2pt: 17 clips
local /content/clips_local | 3pt: 0 clips
Drive clips folder | 2pt: 63 clips
Drive clips folder | 3pt: 53 clips


In [ ]:
# import os
# print("2pt done:", len(os.listdir('/content/clips_local/2pt')))
# print("3pt done:", len(os.listdir('/content/clips_local/3pt')))

2pt done: 17
3pt done: 0


In [ ]:
# # Push finished clips from local disk back to Drive
# DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/CourtVision/clips"
# shutil.copytree(LOCAL_OUTPUT_DIR, DRIVE_OUTPUT_DIR, dirs_exist_ok=True)
# print("Clips copied to Drive:", DRIVE_OUTPUT_DIR)

NameError: name 'LOCAL_OUTPUT_DIR' is not defined

In [ ]:
# ---- CELL 7: Quick sanity check — count clips per class ----

for label in ("2pt", "3pt"):
    folder = os.path.join(OUTPUT_DIR, label)
    count = len([f for f in os.listdir(folder) if f.endswith(".mp4")])
    print(f"{label}: {count} clips")


2pt: 63 clips
3pt: 53 clips
